<a href="https://colab.research.google.com/github/digitalirrigation-lgtm/Awash-river/blob/main/Awash%20river.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee, geemap
ee.Authenticate(force=True)

ee.Initialize(project='irrigation-intelligence-pro')
print("Earth Engine ready for Project 3")

In [ ]:
import ee
import geemap

ee.Initialize(project='irrigation-intelligence-pro')

# 1. Afar boundary (Using simplified 500m version so it loads much faster without timing out)
gaul = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2025/level1')
aoi = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia')) \
          .filter(ee.Filter.eq('GAUL1_NAME', 'Afar')) \
          .geometry()

# 2. Load Sentinel-1
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(aoi) \
    .filterDate('2023-10-01', '2025-04-01') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .select('VV')

# 3. Filter for Dry Season (Oct–Mar)
dry_filter = (
    ee.Filter.calendarRange(10, 12, 'month')
    .Or(ee.Filter.calendarRange(1, 3, 'month'))
)
s1_dry = s1.filter(dry_filter)

print("Dry-season images:", s1_dry.size().getInfo())

# 4. Create a CLEAR image (Temporal Median + Speckle Smoothing)
clean_s1 = (
    s1_dry.median()                       # Removes temporal noise across all dry-season passes
    .focal_median(30, 'circle', 'meters') # Smooths radar speckle graininess
    .clip(aoi)                            # Clips cleanly to Afar's borders
)

# 5. Display the clear image on an interactive map in Colab
Map = geemap.Map()
Map.centerObject(aoi, 7)

# Visualization parameters for Sentinel-1 VV backscatter (in dB)
vis_params = {
    'min': -25,
    'max': 0,
    'palette': ['000000', '404040', '808080', 'bfbfbf', 'ffffff']
}

Map.addLayer(clean_s1, vis_params, 'Clear Sentinel-1 VV (Afar Dry Season)')
Map


In [ ]:
import ee
import requests
from IPython.display import Image, display

ee.Initialize(project='irrigation-intelligence-pro')

# 1. Afar boundary
aoi = (
    ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2025/level1')
    .filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
    .filter(ee.Filter.eq('GAUL1_NAME', 'Afar'))
    .geometry()
)

# 2. Load & filter Sentinel-1 Dry Season (Oct-Mar)
dry_filter = (
    ee.Filter.calendarRange(10, 12, 'month')
    .Or(ee.Filter.calendarRange(1, 3, 'month'))
)

s1_dry = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate('2023-10-01', '2025-04-01')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    .select('VV')
    .filter(dry_filter)
)

# 3. Create a clear, speckle-smoothed image clipped strictly to Afar
clean_s1 = (
    s1_dry.median()
    .focal_median(30, 'circle', 'meters')
    .clip(aoi)
)

# 4. Render ONLY the standalone image (everything outside Afar is transparent)
url = clean_s1.getThumbURL({
    'min': -25,
    'max': 0,
    'dimensions': 1000,
    'region': aoi,
    'format': 'png'
})

display(Image(url=url))


In [ ]:
import ee
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

ee.Initialize(project='irrigation-intelligence-pro')

# 1. Afar boundary (contains the lower & middle Awash River basin)
aoi = (
    ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2025/level1')
    .filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia'))
    .filter(ee.Filter.eq('GAUL1_NAME', 'Afar'))
    .geometry()
)

# 2. Load Sentinel-1 VV Descending
s1 = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    .select('VV')
)

# 3. Separate Dry Season 2024 (Oct 2023 - Mar 2024) & Dry Season 2025 (Oct 2024 - Mar 2025)
s1_2024 = s1.filterDate('2023-10-01', '2024-04-01').median().focal_median(30, 'circle', 'meters')
s1_2025 = s1.filterDate('2024-10-01', '2025-04-01').median().focal_median(30, 'circle', 'meters')

# 4. Calculate 2-Year Change (2025 minus 2024 in dB)
diff = s1_2025.subtract(s1_2024).clip(aoi)

# 5. Classify into distinct, easy-to-read visible colors
# 1: Major Decrease / Drying (Blue) -> < -2.5 dB
# 2: Moderate Decrease (Cyan) -> -2.5 to -1.0 dB
# 3: Stable / No Change (Light Gray) -> -1.0 to +1.0 dB
# 4: Moderate Increase / Crop Growth (Yellow-Orange) -> +1.0 to +2.5 dB
# 5: Major Increase / New Irrigation or Wetting (Bright Red) -> > +2.5 dB
classified_change = (
    ee.Image(3)
    .where(diff.lt(-2.5), 1)
    .where(diff.gte(-2.5).And(diff.lt(-1.0)), 2)
    .where(diff.gt(1.0).And(diff.lte(2.5)), 4)
    .where(diff.gt(2.5), 5)
    .clip(aoi)
)

palette = [
    '#08519c',  # 1: Deep Blue (Strong Drying / Water Loss)
    '#6baed6',  # 2: Light Blue (Moderate Drying / Reduced Vegetation)
    '#e0e0e0',  # 3: Light Gray (Stable / Unchanged)
    '#fe9929',  # 4: Orange (Moderate Growth / Increased Moisture)
    '#d7191c'   # 5: Bright Red (Strong Increase / New Irrigation or Flooding)
]

# 6. Fetch pure PNG image from Earth Engine
url = classified_change.getThumbURL({
    'min': 1,
    'max': 5,
    'palette': palette,
    'dimensions': 1200,
    'region': aoi,
    'format': 'png'
})

response = requests.get(url)
img = Image.open(BytesIO(response.content))

# 7. Plot clean image with a clear Legend (no axes or extra clutter)
fig, ax = plt.subplots(figsize=(10, 10), dpi=150)
ax.imshow(img)
ax.axis('off')

legend_items = [
    mpatches.Patch(color='#d7191c', label='Strong Increase (> +2.5 dB): New Irrigation / Wetting'),
    mpatches.Patch(color='#fe9929', label='Moderate Increase (+1 to +2.5 dB): Crop / Vegetation Growth'),
    mpatches.Patch(color='#e0e0e0', label='Stable (-1 to +1 dB): Unchanged Land / Desert'),
    mpatches.Patch(color='#6baed6', label='Moderate Decrease (-2.5 to -1 dB): Drying / Reduced Cover'),
    mpatches.Patch(color='#08519c', label='Strong Decrease (< -2.5 dB): Water Loss / Harvested Fields')
]

ax.legend(
    handles=legend_items,
    loc='lower left',
    frameon=True,
    facecolor='white',
    edgecolor='black',
    fontsize=9,
    title='Awash River / Afar 2-Year Dry Season Change (2024 vs 2025)',
    title_fontsize=10
)

plt.tight_layout()
plt.savefig('afar_awash_change_map.png', bbox_inches='tight', pad_inches=0.1)
plt.show()


In [ ]:
# --- EXPORT AREA STATISTICS TO GOOGLE DRIVE AS CSV ---

# 1. Calculate pixel area in square meters, convert to hectares (1 ha = 10,000 m²)
pixel_area_ha = ee.Image.pixelArea().divide(10000)

# 2. Group area by each change class (1 to 5)
area_stats = pixel_area_ha.addBands(classified_change.rename('class')).reduceRegion(
    reducer=ee.Reducer.sum().group(
        groupField=1,
        groupName='class_id'
    ),
    geometry=aoi,
    scale=100,        # 100m resolution for fast, accurate regional stats
    maxPixels=1e13
)

# 3. Format the results into an Earth Engine FeatureCollection for CSV export
groups = ee.List(area_stats.get('groups'))

class_names = ee.Dictionary({
    '1': 'Strong Decrease (< -2.5 dB): Water Loss / Harvested',
    '2': 'Moderate Decrease (-2.5 to -1.0 dB): Drying',
    '3': 'Stable (-1.0 to +1.0 dB): Unchanged',
    '4': 'Moderate Increase (+1.0 to +2.5 dB): Crop Growth',
    '5': 'Strong Increase (> +2.5 dB): New Irrigation / Wetting'
})

def format_row(item):
    d = ee.Dictionary(item)
    cid = ee.Number(d.get('class_id')).format('%d')
    area_ha = ee.Number(d.get('sum'))
    area_km2 = area_ha.divide(100)
    return ee.Feature(None, {
        'Class_ID': cid,
        'Change_Category': class_names.get(cid),
        'Area_Hectares': area_ha,
        'Area_SqKm': area_km2
    })

stats_fc = ee.FeatureCollection(groups.map(format_row))

# 4. Export CSV to Google Drive (Folder: EarthEngine_Exports)
task = ee.batch.Export.table.toDrive(
    collection=stats_fc,
    description='Afar_Awash_Change_Stats_2024_2025',
    folder='EarthEngine_Exports',
    fileNamePrefix='afar_awash_change_stats',
    fileFormat='CSV'
)
task.start()
print("Export started! Check your Google Drive -> 'EarthEngine_Exports' folder in ~1-2 minutes.")


In [ ]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Load the GAUL 2025 Level 1 (Region) dataset
gaul = ee.FeatureCollection('FAO/GAUL/2025/level1')

# 2. Filter for Ethiopia and the Afar Region
afar = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia')) \
           .filter(ee.Filter.eq('GAUL1_NAME', 'Afar'))

# 3. Extract the pure geometry
aoi = afar.geometry()

# 4. Calculate the area to verify it is correct
area_sqkm = aoi.area().divide(1e6).getInfo()
print(f"Afar Region Area: {area_sqkm:,.0f} km²")

In [ ]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Rebuild boundary (self-contained)
gaul = ee.FeatureCollection('FAO/GAUL/2025/level1')
aoi = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia')) \
          .filter(ee.Filter.eq('GAUL1_NAME', 'Afar')).geometry()

# 2. Load Sentinel-1 Radar Data
s1 = ee.ImageCollection('COPERNICUS/S1_GRD')

# 3. Filter by Boundary, Date, and Radar Settings
s1_filtered = s1.filterBounds(aoi) \
                .filterDate('2024-01-01', '2025-12-31') \
                .filter(ee.Filter.eq('instrumentMode', 'IW')) \
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
                .select('VV')

# 4. Count how many radar images we have
count = s1_filtered.size().getInfo()
print(f"Number of Sentinel-1 radar images over Afar (2024-2025): {count}")

In [ ]:
import ee
ee.Initialize(project='irrigation-intelligence-pro')

# 1. Rebuild boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level1')
aoi = gaul.filter(ee.Filter.eq('GAUL0_NAME', 'Ethiopia')) \
          .filter(ee.Filter.eq('GAUL1_NAME', 'Afar')).geometry()

# 2. Load Sentinel-1 and apply strict filters
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(aoi) \
        .filterDate('2024-01-01', '2025-12-31') \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
        .select('VV')

# 3. Keep only Dry Season months (Oct, Nov, Dec, Jan, Feb, Mar)
def is_dry_season(img):
    month = ee.Date(img.get('system:time_start')).get('month')
    return img.set('dry', month.gte(10).Or(month.lte(3)))

s1_dry = s1.map(is_dry_season).filter(ee.Filter.eq('dry', True))

# 4. Speckle Filter and Water Threshold (< -20 dB)
def create_water_mask(img):
    # Focal median removes the "salt and pepper" noise of radar
    smoothed = img.focal_median(3, 'square', 'pixels')
    # If the signal is less than -20, it is water
    water = smoothed.lt(-20).rename('water')
    return water.copyProperties(img, ['system:time_start'])

water_masks = s1_dry.map(create_water_mask)

# 5. Create the 2024 and 2025 Composites
# We use median() to get the most persistent water bodies
water_2024 = water_masks.filterDate('2024-01-01', '2024-12-31').median()
water_2025 = water_masks.filterDate('2025-01-01', '2025-12-31').median()

print(f"Total dry-season images used: {s1_dry.size().getInfo()}")
print("Water masks for 2024 and 2025 are ready!")

In [ ]:

import ee

# 1. Connect to Earth Engine
ee.Initialize(project='irrigation-intelligence-pro')

# 2. Load Afar boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level1')

afar = gaul.filter(
    ee.Filter.eq('GAUL0_NAME', 'Ethiopia')
).filter(
    ee.Filter.eq('GAUL1_NAME', 'Afar')
)

aoi = afar.geometry()

print("Afar boundary count:", afar.size().getInfo())
print("Afar area (km²):", round(
    aoi.area().divide(1e6).getInfo(), 2
))

# 3. Load Sentinel-1 without filters
s1_all = ee.ImageCollection('COPERNICUS/S1_GRD')

print("\n--- SENTINEL-1 FILTER DIAGNOSTICS ---")

# Count images intersecting Afar
s1_aoi = s1_all.filterBounds(aoi)
print("1. Images intersecting Afar:",
      s1_aoi.size().getInfo())

# Date filter
s1_date = s1_aoi.filterDate(
    '2024-01-01', '2026-01-01'
)
print("2. Images in date range:",
      s1_date.size().getInfo())

# Instrument mode
s1_iw = s1_date.filter(
    ee.Filter.eq('instrumentMode', 'IW')
)
print("3. IW mode:",
      s1_iw.size().getInfo())

# VV polarization
s1_vv = s1_iw.filter(
    ee.Filter.listContains(
        'transmitterReceiverPolarisation', 'VV'
    )
)
print("4. VV polarization:",
      s1_vv.size().getInfo())

# Descending orbit
s1_desc = s1_vv.filter(
    ee.Filter.eq('orbitProperties_pass', 'DESCENDING')
)
print("5. Descending orbit:",
      s1_desc.size().getInfo())

# 4. Check available dates and orbit directions
if s1_aoi.size().getInfo() > 0:

    first = ee.Image(s1_aoi.first())

    print("\nFirst image date:",
          ee.Date(first.get('system:time_start'))
          .format('YYYY-MM-dd').getInfo())

    print("Available polarizations:",
          first.get('transmitterReceiverPolarisation').getInfo())

    print("Available instrument mode:",
          first.get('instrumentMode').getInfo())

    print("Available orbit direction:",
          first.get('orbitProperties_pass').getInfo())

else:
    print("\nNo Sentinel-1 images intersect the Afar boundary.")

# 5. Test dry-season filtering
def add_month_property(img):
    month = ee.Date(
        img.get('system:time_start')
    ).get('month')

    return img.set(
        'dry',
        month.gte(10).Or(month.lte(3))
    )

s1_dry_test = s1_desc.map(
    add_month_property
).filter(
    ee.Filter.eq('dry', True)
)

print("\n6. Dry-season images:",
      s1_dry_test.size().getInfo())


In [ ]:

import ee

ee.Initialize(project='irrigation-intelligence-pro')

# 1. Afar boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level1')

aoi = gaul.filter(
    ee.Filter.eq('GAUL0_NAME', 'Ethiopia')
).filter(
    ee.Filter.eq('GAUL1_NAME', 'Afar')
).geometry()

# 2. Load Sentinel-1
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(aoi) \
    .filterDate('2023-10-01', '2025-04-01') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains(
        'transmitterReceiverPolarisation', 'VV'
    )) \
    .filter(ee.Filter.eq(
        'orbitProperties_pass', 'DESCENDING'
    )) \
    .select('VV')

# 3. Select October, November, December,
# January, February and March
dry_filter = (
    ee.Filter.calendarRange(10, 12, 'month')
    .Or(ee.Filter.calendarRange(1, 3, 'month'))
)

s1_dry = s1.filter(dry_filter)

# 4. Count images
print("Total Sentinel-1 images:", s1.size().getInfo())

print("Dry-season images:",
      s1_dry.size().getInfo())

# 5. Check the two seasonal periods separately
season_2024 = s1_dry.filterDate(
    '2023-10-01', '2024-04-01'
)

season_2025 = s1_dry.filterDate(
    '2024-10-01', '2025-04-01'
)

print("Dry season 2024:",
      season_2024.size().getInfo())

print("Dry season 2025:",
      season_2025.size().getInfo())


In [ ]:

import ee
import geemap

ee.Initialize(project='irrigation-intelligence-pro')

# 1. Rebuild Afar boundary
gaul = ee.FeatureCollection('FAO/GAUL/2025/level1')

aoi = gaul.filter(
    ee.Filter.eq('GAUL0_NAME', 'Ethiopia')
).filter(
    ee.Filter.eq('GAUL1_NAME', 'Afar')
).geometry()

# 2. Load the two dry seasons
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(aoi) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains(
        'transmitterReceiverPolarisation', 'VV'
    )) \
    .filter(ee.Filter.eq(
        'orbitProperties_pass', 'DESCENDING'
    )) \
    .select('VV')

dry_filter = (
    ee.Filter.calendarRange(10, 12, 'month')
    .Or(ee.Filter.calendarRange(1, 3, 'month'))
)

season_2024 = s1.filterDate(
    '2023-10-01', '2024-04-01'
).filter(dry_filter)

season_2025 = s1.filterDate(
    '2024-10-01', '2025-04-01'
).filter(dry_filter)

# 3. Speckle filtering and water detection
def extract_water(img):

    # Smooth radar speckle
    smooth = img.focalMedian(
        radius=3,
        kernelType='square',
        units='pixels'
    )

    # Initial water threshold: below -20 dB
    water = smooth.lt(-20).rename('water')

    return water.copyProperties(
        img, ['system:time_start']
    )

# 4. Apply water extraction to every image
water_2024 = season_2024.map(extract_water)
water_2025 = season_2025.map(extract_water)

# 5. Calculate water occurrence
# Mean of binary images = fraction of observations
# classified as water

occurrence_2024 = water_2024.mean().rename(
    'water_occurrence'
)

occurrence_2025 = water_2025.mean().rename(
    'water_occurrence'
)

# 6. Identify persistent water
# At least 50% of valid observations classified as water

persistent_2024 = occurrence_2024.gte(0.5).selfMask()
persistent_2025 = occurrence_2025.gte(0.5).selfMask()

# 7. Print results
print("2024 images:", season_2024.size().getInfo())
print("2025 images:", season_2025.size().getInfo())

print("2024 water extraction completed.")
print("2025 water extraction completed.")

# 8. Display maps
Map = geemap.Map()

Map.centerObject(aoi, 7)

Map.addLayer(
    occurrence_2024,
    {'min': 0, 'max': 1,
     'palette': ['white', 'cyan', 'blue']},
    '2024 Water Occurrence'
)

Map.addLayer(
    occurrence_2025,
    {'min': 0, 'max': 1,
     'palette': ['white', 'yellow', 'red']},
    '2025 Water Occurrence'
)

Map.addLayer(
    persistent_2024,
    {'palette': ['0000FF']},
    '2024 Persistent Water'
)

Map.addLayer(
    persistent_2025,
    {'palette': ['FF0000']},
    '2025 Persistent Water'
)

Map.addLayer(aoi, {}, 'Afar Boundary', False)

Map.addLayerControl()
Map


In [ ]:
import geemap
import ee

ee.Initialize(project='irrigation-intelligence-pro')

Map = geemap.Map()
Map.setCenter(40.5, 11.5, 7)

# Display a simple satellite image
test_image = ee.Image('COPERNICUS/S2_SR_HARMONIZED').first()

Map.addLayer(
    test_image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'Satellite test'
)

Map